# v7 DSL Dataset Preparation

This notebook scaffolds the split-aware SVG/DSL dataset workflow around `version/v7/data/spec04` and the existing `v7` dataset tools.

Use it to:
- inspect the seed workspace and manifests
- materialize SVG stage artifacts into the workspace
- stage the workspace into a run-local `dataset/` snapshot
- regenerate `dataset_viewer.html` and refresh `ir_hub.html`
- hand off the staged dataset workspace into the `ck_run_v7.py init` and `TrainingProject.materialize()` training paths

Launch from the repo root:
- `.venv/bin/jupyter lab notebooks/python_authoring/v7_training/03_v7_dsl_dataset_preparation.ipynb`

This notebook is a scaffold. The heavy commands are wired up, but they do not execute until you flip the `EXECUTE_*` flags in the execution cell.


In [1]:
from pathlib import Path
import json
import shlex
import subprocess
import sys
from IPython.display import HTML, display

REPO_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "ckernel_engine").exists() and (candidate / "version" / "v7").exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise RuntimeError(
        f"Could not find the C-Kernel-Engine repo root from cwd={Path.cwd()}. "
        "Open this notebook from inside the repo checkout."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

PYTHON_EXEC = Path(sys.executable).resolve()
V7_ROOT = REPO_ROOT / "version" / "v7"
print(f"Repo root: {REPO_ROOT}")
print(f"Python exec: {PYTHON_EXEC}")


Repo root: /home/antshiv/Workspace/C-Kernel-Engine
Python exec: /usr/bin/python3.12


In [2]:
def shell_join(command):
    return shlex.join(str(part) for part in command)


def read_json(path):
    path = Path(path)
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))


def run_and_print(command, *, check=True):
    rendered = [str(part) for part in command]
    print("$ " + shell_join(rendered))
    completed = subprocess.run(
        rendered,
        cwd=str(REPO_ROOT),
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout.rstrip())
    if completed.stderr:
        print(completed.stderr.rstrip())
    if check and completed.returncode != 0:
        raise RuntimeError(f"command failed with exit code {completed.returncode}")
    return completed


def existing_uri(path):
    path = Path(path)
    return path.resolve().as_uri() if path.exists() else None


In [3]:
WORKSPACE = REPO_ROOT / "version" / "v7" / "data" / "spec04"
RUN_DIR = Path.home() / ".cache" / "ck-engine-v7" / "models" / "train" / "v7-dsl-dataset-notebook-demo"
DATASET_ROOT = RUN_DIR / "dataset"
DATASET_VIEWER = RUN_DIR / "dataset_viewer.html"
DATASET_SNAPSHOT = DATASET_ROOT / "dataset_snapshot.json"
MODELS_ROOT = RUN_DIR.parent.parent

{
    "workspace": str(WORKSPACE),
    "run_dir": str(RUN_DIR),
    "dataset_root": str(DATASET_ROOT),
    "dataset_viewer": str(DATASET_VIEWER),
    "models_root": str(MODELS_ROOT),
}


{'workspace': '/home/antshiv/Workspace/C-Kernel-Engine/version/v7/data/spec04',
 'run_dir': '/home/antshiv/.cache/ck-engine-v7/models/train/v7-dsl-dataset-notebook-demo',
 'dataset_root': '/home/antshiv/.cache/ck-engine-v7/models/train/v7-dsl-dataset-notebook-demo/dataset',
 'dataset_viewer': '/home/antshiv/.cache/ck-engine-v7/models/train/v7-dsl-dataset-notebook-demo/dataset_viewer.html',
 'models_root': '/home/antshiv/.cache/ck-engine-v7/models'}

## Inspect The Seed Workspace

The split-aware seed workspace lives under `version/v7/data/spec04`. It is the repo template, not the final operator artifact home.

Once you stage it into a run, the working dataset copy, `dataset_viewer.html`, `ir_report.html`, checkpoints, and training outputs should all live together under the run directory.


In [4]:
workspace_entries = sorted(p.name for p in WORKSPACE.iterdir()) if WORKSPACE.exists() else []
manifest_dir = WORKSPACE / "manifests"
manifest_summaries = {}
if manifest_dir.exists():
    for path in sorted(manifest_dir.glob("*.json")):
        payload = read_json(path)
        if isinstance(payload, dict):
            manifest_summaries[path.name] = payload.get("schema") or payload.get("format") or sorted(payload.keys())[:5]
        else:
            manifest_summaries[path.name] = "unreadable"

{
    "workspace_exists": WORKSPACE.exists(),
    "workspace_entries": workspace_entries,
    "manifests": manifest_summaries,
}


{'workspace_exists': True,
 'workspace_entries': ['README.md',
  'contracts',
  'holdout',
  'manifests',
  'midtrain',
  'normalized',
  'pretrain',
  'raw_assets',
  'sft',
  'tokenizer'],
 'manifests': {}}

## Materialize And Stage The Workspace

The notebook keeps the command surface explicit.

The main workflow is:
1. one-time workspace scaffold: `init_data_workspace_v7.sh`
2. refresh workspace artifacts: `materialize_svg_stage_artifacts_v7.py`
3. stage the workspace into a run and build `dataset_viewer.html`: `stage_dataset_workspace_v7.py`
4. refresh the shared `ir_hub.html`: `open_ir_hub.py`

The next cell builds those commands. The execution cell after that is safe by default and will only run them if you set the flags to `True`.


In [5]:
INIT_WORKSPACE_CMD = [
    "bash",
    str(REPO_ROOT / "version" / "v7" / "scripts" / "init_data_workspace_v7.sh"),
    "--spec",
    "spec04",
    "--dataset-type",
    "svg",
]

MATERIALIZE_WORKSPACE_CMD = [
    str(PYTHON_EXEC),
    str(REPO_ROOT / "version" / "v7" / "scripts" / "materialize_svg_stage_artifacts_v7.py"),
    "--workspace",
    str(WORKSPACE),
    "--force",
]

STAGE_WORKSPACE_CMD = [
    str(PYTHON_EXEC),
    str(REPO_ROOT / "version" / "v7" / "scripts" / "dataset" / "stage_dataset_workspace_v7.py"),
    "--workspace",
    str(WORKSPACE),
    "--run-dir",
    str(RUN_DIR),
    "--mode",
    "copy",
    "--force",
]

REBUILD_VIEWER_CMD = [
    str(PYTHON_EXEC),
    str(REPO_ROOT / "version" / "v7" / "scripts" / "build_svg_dataset_visualizer_v7.py"),
    "--workspace",
    str(DATASET_ROOT),
    "--output",
    str(DATASET_VIEWER),
]

REFRESH_HUB_CMD = [
    str(PYTHON_EXEC),
    str(REPO_ROOT / "version" / "v7" / "tools" / "open_ir_hub.py"),
    "--models-root",
    str(MODELS_ROOT),
    "--output",
    str(MODELS_ROOT / "ir_hub.html"),
    "--index-out",
    str(MODELS_ROOT / "runs_hub_index.json"),
]

{
    "init_workspace": shell_join(INIT_WORKSPACE_CMD),
    "materialize_workspace": shell_join(MATERIALIZE_WORKSPACE_CMD),
    "stage_workspace": shell_join(STAGE_WORKSPACE_CMD),
    "rebuild_viewer": shell_join(REBUILD_VIEWER_CMD),
    "refresh_hub": shell_join(REFRESH_HUB_CMD),
}


{'init_workspace': 'bash /home/antshiv/Workspace/C-Kernel-Engine/version/v7/scripts/init_data_workspace_v7.sh --spec spec04 --dataset-type svg',
 'materialize_workspace': '/usr/bin/python3.12 /home/antshiv/Workspace/C-Kernel-Engine/version/v7/scripts/materialize_svg_stage_artifacts_v7.py --workspace /home/antshiv/Workspace/C-Kernel-Engine/version/v7/data/spec04 --force',
 'stage_workspace': '/usr/bin/python3.12 /home/antshiv/Workspace/C-Kernel-Engine/version/v7/scripts/dataset/stage_dataset_workspace_v7.py --workspace /home/antshiv/Workspace/C-Kernel-Engine/version/v7/data/spec04 --run-dir /home/antshiv/.cache/ck-engine-v7/models/train/v7-dsl-dataset-notebook-demo --mode copy --force',
 'rebuild_viewer': '/usr/bin/python3.12 /home/antshiv/Workspace/C-Kernel-Engine/version/v7/scripts/build_svg_dataset_visualizer_v7.py --workspace /home/antshiv/.cache/ck-engine-v7/models/train/v7-dsl-dataset-notebook-demo/dataset --output /home/antshiv/.cache/ck-engine-v7/models/train/v7-dsl-dataset-note

In [6]:
EXECUTE_MATERIALIZE_WORKSPACE = False
EXECUTE_STAGE_WORKSPACE = False
EXECUTE_REFRESH_HUB = False

if EXECUTE_MATERIALIZE_WORKSPACE:
    run_and_print(MATERIALIZE_WORKSPACE_CMD)

if EXECUTE_STAGE_WORKSPACE:
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    run_and_print(STAGE_WORKSPACE_CMD)
    if not DATASET_VIEWER.exists():
        print("dataset_viewer.html was not created during staging; try REBUILD_VIEWER_CMD explicitly.")

if EXECUTE_REFRESH_HUB:
    run_and_print(REFRESH_HUB_CMD)

if not any([EXECUTE_MATERIALIZE_WORKSPACE, EXECUTE_STAGE_WORKSPACE, EXECUTE_REFRESH_HUB]):
    print("Set one or more EXECUTE_* flags to True to run the scaffolded dataset commands.")


Set one or more EXECUTE_* flags to True to run the scaffolded dataset commands.


## Inspect Staged Dataset Artifacts

After staging, the key files to inspect are:
- `RUN/dataset/dataset_snapshot.json`
- `RUN/dataset/manifests/*workspace_manifest.json`
- `RUN/dataset_viewer.html`
- `MODELS_ROOT/ir_hub.html`

If `ir_report.html` exists too, that means you already handed the run off into the training/materialization path.


In [7]:
snapshot = read_json(DATASET_SNAPSHOT)
workspace_manifest_candidates = sorted((DATASET_ROOT / "manifests").glob("*workspace_manifest.json")) if DATASET_ROOT.exists() else []
workspace_manifest_path = workspace_manifest_candidates[0] if workspace_manifest_candidates else None
link_paths = [
    ("Dataset Viewer", DATASET_VIEWER),
    ("Dataset Snapshot", DATASET_SNAPSHOT),
    ("Workspace Manifest", workspace_manifest_path),
    ("IR Report", RUN_DIR / "ir_report.html"),
    ("IR Hub", MODELS_ROOT / "ir_hub.html"),
]
display(
    HTML(
        "<ul>"
        + "".join(
            f"<li><a href='{Path(path).resolve().as_uri()}'>{label}</a></li>"
            for label, path in link_paths
            if path is not None and Path(path).exists()
        )
        + "</ul>"
    )
)

{
    "dataset_viewer_exists": DATASET_VIEWER.exists(),
    "dataset_snapshot_exists": DATASET_SNAPSHOT.exists(),
    "workspace_manifest": str(workspace_manifest_path) if workspace_manifest_path else None,
    "staged_entries": snapshot.get("staged_entries", []) if isinstance(snapshot, dict) else None,
    "missing_entries": snapshot.get("missing_entries", []) if isinstance(snapshot, dict) else None,
}


{'dataset_viewer_exists': False,
 'dataset_snapshot_exists': False,
 'workspace_manifest': None,
 'staged_entries': None,
 'missing_entries': None}

## Handoff Into Training

This notebook stops at dataset preparation, but it should also show the next hop into the real v7 training surface.

Two handoff styles matter:
- Python authoring path: `TrainingProject(...).materialize(MaterializeOptions(dataset_workspace=...))`
- CLI path: `ck_run_v7.py init --dataset-workspace ... --dataset-stage-force ...`

The cell below builds both without running a full train. It also surfaces the stage-specific train files that the workspace currently exposes.


In [8]:
from ckernel_engine.v7 import MaterializeOptions, TemplateSpec, TinyModelSpec, TokenizerPlan, TrainingProject

train_data_candidates = {
    "pretrain_train": [str(path) for path in sorted((WORKSPACE / "pretrain" / "train").glob("*.txt"))[:5]],
    "midtrain_train": [str(path) for path in sorted((WORKSPACE / "midtrain" / "train").glob("*.txt"))[:5]],
    "sft_train": [str(path) for path in sorted((WORKSPACE / "sft" / "train").glob("*.txt"))[:5]],
}

project = TrainingProject(
    run_name=RUN_DIR.name,
    run_dir=RUN_DIR,
    model=TinyModelSpec(
        init="xavier_uniform",
        layers=2,
        vocab_size=256,
        embed_dim=128,
        hidden_dim=256,
        num_heads=8,
        num_kv_heads=4,
        context_len=128,
    ),
    template=TemplateSpec.builtin_template("qwen3"),
    tokenizer=TokenizerPlan(
        family="runtime_default",
        notes="Dataset notebook handoff example.",
    ),
)

materialize_options = MaterializeOptions(
    generate_ir=True,
    generate_runtime=True,
    strict=True,
    dataset_workspace=WORKSPACE,
    dataset_stage_mode="copy",
    dataset_stage_force=True,
)

cli_init_cmd = [
    str(PYTHON_EXEC),
    str(REPO_ROOT / "version" / "v7" / "scripts" / "ck_run_v7.py"),
    "init",
    "--run",
    str(RUN_DIR),
    "--run-name",
    RUN_DIR.name,
    "--template",
    "qwen3",
    "--generate-ir",
    "--generate-runtime",
    "--strict",
    "--dataset-workspace",
    str(WORKSPACE),
    "--dataset-stage-mode",
    "copy",
    "--dataset-stage-force",
    *project.model.to_init_args(),
]

{
    "python_materialize_options": materialize_options.to_metadata(),
    "cli_init_command": shell_join(cli_init_cmd),
    "pretrain_train_files": train_data_candidates["pretrain_train"],
    "midtrain_train_files": train_data_candidates["midtrain_train"],
    "sft_train_files": train_data_candidates["sft_train"],
}


{'python_materialize_options': {'generate_ir': True,
  'generate_runtime': True,
  'strict': True,
  'train_mode': 'pretrain',
  'bridge_lowering': 'legacy',
  'checkpoint_policy': 'none',
  'dataset_workspace': '/home/antshiv/Workspace/C-Kernel-Engine/version/v7/data/spec04',
  'dataset_stage_mode': 'copy',
  'dataset_stage_force': True},
 'cli_init_command': '/usr/bin/python3.12 /home/antshiv/Workspace/C-Kernel-Engine/version/v7/scripts/ck_run_v7.py init --run /home/antshiv/.cache/ck-engine-v7/models/train/v7-dsl-dataset-notebook-demo --run-name v7-dsl-dataset-notebook-demo --template qwen3 --generate-ir --generate-runtime --strict --dataset-workspace /home/antshiv/Workspace/C-Kernel-Engine/version/v7/data/spec04 --dataset-stage-mode copy --dataset-stage-force --train-seed 42 --init xavier_uniform --layers 2 --vocab-size 256 --embed-dim 128 --hidden-dim 256 --num-heads 8 --num-kv-heads 4 --context-len 128 --rope-theta 1000000.0 --kernel-policy fp32_reference_first --adamw-beta1 0.9 -